# 00 · The Monthly Stats data — what it is, and how to read it

This tutorial will help you become more familiar with the M-Lab monthly stats dataset. This dataset is an accessable summary of measurement data at various aggregations from the petabytes of Internet measurements tests that M-Lab makes available and curates. This is a great starting point for exploring our Internet measurements.  

After going through these notebooks, you will be able to say what the Monthly Stats dataset is, what its four metrics mean, how to read a percentile, and which pieces of the data exist for your own country, subdivisions (states, provinces, districs), and ASN (unique ISP number).

**The plan for this tutorial**

| Notebook | Question it answers |
|---|---|
| **00** · this notebook | What are these data, and how do I load them? |
| **01** · country explorer | How do I explore the data aggregated at the country level and make comparisons? |
| **02** · the splits | How do these measures vary: between countries, regions, cities, or Internet providers? |
| **03** · multiple months | How do measurements vary across months / countries / cities / ISPs? |

## What is the Monthly Stats dataset?

[M-Lab](https://www.measurementlab.net/) runs an open speed-test platform used millions of times per day from real user devices worldwide. We aggregate those NDT results into **monthly, percentile-based summaries** at several geographic and Internet topological granularities.

Instead of querying millions or billions of raw rows of speed-test data in our archive, with the `monthlystats` dataset each month you download one small file (several megabytes) that already contains pre-computed percentile values for each metric, for each geography, for the month in question.

> The Internet Quality Barometer (IQB) project builds on this dataset — see the [IQB publications](https://www.measurementlab.net/publications/IQB_report_2025.pdf) for that composite-score view. Here we stay with the metrics themselves.

## The four metrics

| Metric | Column prefix | Unit | Better connectivity |
|---|---|---|---|
| Download throughput | `download_p*` | Mbit/s | ↑ Higher is better|
| Upload throughput | `upload_p*` | Mbit/s | ↑ Higher is better |
| Latency (min RTT) | `latency_p*` | ms | ↓ Lower is better |
| Packet loss rate | `loss_p*` | fraction 0–1 | ↓ Lower is better |

- **download** is how fast pages and files arrive at your computer
- **upload** is how fast you send data away from your computer
- **latency** is round-trip responsiveness, how long it takes for a message to get to the server and be acknowledged at your computer
- **loss** is a percentage of dropped packets (reliability)

> **Percentile polarity** — for latency and loss, *lower* is better, so the data **flip the percentiles**: in this dataset, a higher percentile means a *better* connection. `latency_p95` is the 5% of connections with the lowest (best) latency; `latency_p5` is the slowest (worst) latency. Notebook 01 makes this visible on real curves.

## Percentiles — line everyone up

A percentile comes from **ordering all the tests up in a month, slowest to fastest:**

- **p50** (the median) is the test in the exact middle of all tests;
- **p95** is 95th position out of 100 — near the front of the line;
- **p1** is near the very back.

There is no mean (average) in this dataset, only percentiles. One very fast connection does not drag the summary upward the way it drags an average.

A **wide p50→p95 gap shows how different experineces can be in that region / country** — it is a statement about the *shape* of the distribution. Most tests sit in a band, and a minority run much faster or much slower. Where the median sits, how long the tail stretches, how flat or steep the middle is: that shape is what notebook 01 teaches you to read.

## The splits available in these data

"One row per what" is the only thing that changes between slices; every slice has the same four metrics across their percentiles.

| Slice (prefix) | One row per… | What you can learn with it | Taught in |
|---|---|---|---|
| `by_country` | country | Cross-national benchmarking; how distributions differ between countries | 01 |
| `by_country_subdivision1` | state / province | Geography of quality *within* a country (urban vs rural) | 02 |
| `by_country_city` | city | City-level benchmarking; which cities stand out from their country | 02 |
| `by_country_asn` | provider (ASN) | Provider differences within a country — **handle with care: the data are not cleaned, so we compare, never rank or promote** | 02 |

Upload files mirror the download files: `uploads_by_country`, `uploads_by_country_city`, and so on.

## Dates and coverage

Every notebook in this tutorial derives "the newest month" from the manifest itself, and notebook 02 uses the newest month that exists in *all* the slices it needs. You never type a date by hand.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# 1. Import the libraries the notebook uses.
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 2. Plot style: light grid lines and a muted palette.
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

print("Ready.")

In [ ]:
# ── Load the manifest ─────────────────────────────────────────────────────────
# 1. Download the manifest: a list of every published monthly-stats file.
MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

# 2. Turn it into a table with one row per file: month, end, slice, URL.
records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # Path format: cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))

# 3. Helper: the direct download URL for one month of one slice.
def month_url(slice_name, start):
    # start looks like 'YYYY-MM-DD' (the first day of the month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]

# 4. Helper: the newest month that has data for a slice.
def latest_month(slice_name):
    return catalog.loc[catalog["slice"] == slice_name, "start"].max().strftime("%Y-%m-%d")

print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices,",
      catalog["start"].min().date(), "→", catalog["start"].max().date())

## Look at a sample of the data

Load the newest month of `downloads_by_country` — always derived from the manifest, never hardcoded — and look at the first rows.

In [ ]:
# ── Look at a sample of the data ──────────────────────────────────────────────
# Load the newest month of country-level download data.
MONTH = latest_month("downloads_by_country")
df = pd.read_parquet(month_url("downloads_by_country", MONTH))

# Print a quick summary, then show the first rows so we can see the layout.
print("Newest month:", MONTH)
print("Rows:", len(df), "countries")
print()
print("Columns:", list(df.columns))
print()
df.head()

## Play with these four short recipes

Four short recipes, adapted from an exploratory notebook by Pavlos. Each is the kernel of a much bigger question.

**You can edit these cells and re-run them, play, change some things like the country code `US`, try your own country, or look at different columns `upload_p50` maybe?**

1. The median download for one country, for the newest month.
2. A table comparing several countries on two metrics at once.
3. Filtering a city slice to one city — the same pattern at finer resolution.
4. The full p1–p99 distribution of download speeds for five countries — the shape, in one table.

In [ ]:
# ── Recipe 1 ── the median download for one country.
speed = df[df["country_code"] == "US"]["download_p50"].iloc[0]
print(f"Median download, {MONTH}: {speed:.1f} Mbit/s")

# ── Recipe 2 ── a small table comparing several countries.
# Filter the column 'country_code' to a few countries, pick two columns.
(df[df["country_code"].isin(["US", "DE", "BR", "IN", "NG"])]
    [["country_code", "download_p50", "latency_p50"]]
    .sort_values("download_p50", ascending=False))

In [ ]:
# ── Recipe 3 ── one city, using the fine-grained city slice.
city_month = latest_month("downloads_by_country_city")
city = pd.read_parquet(month_url("downloads_by_country_city", city_month))
london = city[city["city"] == "London"]
print(f"London, {city_month}: {len(london)} rows, "
      f"median download {london['download_p50'].iloc[0]:.0f} Mbit/s "
      f"({london['sample_count'].iloc[0]:,} tests)")

In [ ]:
# ── Recipe 4 ── the full p1–p99 download distribution for five countries.
# Reading across a row is reading that country's distribution shape —
# notebook 01 draws these same numbers as curves.
pcts = [c for c in df.columns if c.startswith("download_p")]
dist = (df[df["country_code"].isin(["US", "BR", "DE", "IN", "NG"])]
        [["country_code"] + pcts]
        .set_index("country_code")
        .sort_values("download_p50", ascending=False))
dist.round(1)

## Explore the catalog

The below code creates a user interface allowing you to pick a slice, then a month: the cell prints the exact download URL and a preview of the first few rows of the table. This is the menu for everything you can ask the data.

In [ ]:
# ── Explore the catalog ───────────────────────────────────────────────────────
# Two dropdowns: choose a slice, then a month within it.
w_slice = widgets.Dropdown(
    options=sorted(catalog["slice"].unique()), value="downloads_by_country",
    description="Slice:", layout=widgets.Layout(width="420px"))
w_month = widgets.Dropdown(description="Month:",
                           layout=widgets.Layout(width="320px"))
out = widgets.Output()

# When the slice changes, repopulate the month dropdown with months that exist.
def on_slice(change=None):
    months = (catalog[catalog["slice"] == w_slice.value]["start"]
              .dt.strftime("%Y-%m-%d").tolist())
    w_month.options = sorted(months, reverse=True)
    w_month.value = months[-1]

# When a month is chosen, print the direct URL and preview the first rows.
def show(change=None):
    with out:
        clear_output(wait=True)
        url = month_url(w_slice.value, w_month.value)
        print("Slice :", w_slice.value)
        print("Month :", w_month.value)
        print("URL   :", url)
        print()
        try:
            preview = pd.read_parquet(url).head()
            print(f"{len(preview)} of {pd.read_parquet(url).shape[0]:,} rows:")
            display(preview)
        except Exception as exc:
            print(f"Could not preview: {exc}")

# Wire the handlers to the widgets and show the controls.
w_slice.observe(on_slice, "value")
w_month.observe(show, "value")
display(widgets.VBox([w_slice, w_month, out]))
on_slice(); show()

## Next steps

- **01 · Country explorer** — read one country's distribution shape; see whether rankings hold at p95.
- **02 · The splits** — regions, cities, and providers inside a country.
- **03 · Multiple months** — trends over time, loaded the same direct way.